# Robustness analyses: media-independent measures and measurement composition

Self-contained companion to the main analysis notebook. It rebuilds the monthly dataset from the raw CSV extracts, then runs the robustness analyses of the paper: the strict Knowledge Graph filter, the taxonomy-drift test and Pielou evenness, the PSNI security index, the Executive-suspension indicator, the dynamic specifications M1–M5, and the Local Projections with the PSNI shock.

**Inputs (place in `data/` or set `BASE`):**
- `events_monthly_northern_ireland.csv`: GDELT Events aggregates
- `kg_theme_counts_northern_ireland.csv`: GKG theme counts (baseline filter)
- `kg_theme_counts_northern_ireland_STRICT.csv`: strict-filter extraction (query below)
- `psni_monthly.csv`: PSNI security series, February 2015 to December 2025

Stormont coding rule and boundary decisions: see `docs/stormont_coding_note.md`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from scipy.stats import entropy

# --- Colab setup -----------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
BASE = "data"       # folder containing the frozen CSV extracts
# BASE = "."                 # local fallback
START, END = "2015-02-01", "2025-12-01"
HAC_MAXLAGS = 3
SEED = 42

def znorm(s):
    return (s - s.mean()) / s.std()

In [ ]:
from pathlib import Path
import os
import sys

# Run from the repository root or notebooks/; an explicit override is optional.
_candidate = Path(os.environ.get("NARRATIVE_REPO_DIR", Path.cwd())).resolve()
REPO_DIR = next((p for p in [_candidate, *_candidate.parents]
                 if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Open the notebook inside the repository or set NARRATIVE_REPO_DIR.")
os.chdir(REPO_DIR)
DATA_DIR = REPO_DIR / "data"
FIGURES_DIR = REPO_DIR / "figures"
BASE = str(DATA_DIR)
IN_COLAB = "google.colab" in sys.modules
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
print(f"Repository: {REPO_DIR}")

RUN_BIGQUERY = False
OVERWRITE_EXISTING = False
GCP_PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_GCP_PROJECT_ID")

RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)


## 1. Rebuild the monthly dataset (identical to the main analysis notebook)


In [ ]:
events = pd.read_csv(f"{BASE}/events_monthly_northern_ireland.csv",
                     parse_dates=["month"]).sort_values("month")
kg = pd.read_csv(f"{BASE}/kg_theme_counts_northern_ireland.csv",
                 parse_dates=["month"]).dropna(subset=["theme"])

events["instability"] = (
    (events["violence"] + events["repression"] + events["security"])
    / events["total_events"].replace(0, np.nan)
).fillna(0)
events["elite_total"] = events["elite_coop"] + events["elite_conflict"]
events["geo_ag_polarization"] = np.where(
    events["elite_total"] > 0,
    events["elite_conflict"] / events["elite_total"], np.nan)

nv = (kg.groupby("month")
        .apply(lambda x: entropy(x["mentions"], base=2))
        .rename("nv").reset_index())
df = events.merge(nv, on="month", how="inner").sort_values("month")
df = df[(df["month"] >= START) & (df["month"] <= END)].reset_index(drop=True)
print(f"{len(df)} months, {df['month'].min():%Y-%m} → {df['month'].max():%Y-%m}")

## 2. PSNI media-independent instability index

In [ ]:
psni = pd.read_csv(f"{BASE}/psni_monthly.csv", parse_dates=["month"])
psni_cols = [c for c in psni.columns if c.startswith("psni_")]
z = (psni[psni_cols] - psni[psni_cols].mean()) / psni[psni_cols].std(ddof=0)
psni["psni_index"] = z.sum(axis=1)
df = df.merge(psni[["month", "psni_index"]], on="month", how="left")

cv = df[["instability", "psni_index"]].dropna()
r, p = stats.pearsonr(cv["instability"], cv["psni_index"])
rho, prho = stats.spearmanr(cv["instability"], cv["psni_index"])
print(f"corr(GDELT instability, PSNI index): "
      f"Pearson r={r:.3f} (p={p:.4f}) | Spearman rho={rho:.3f} (p={prho:.4f})")

plt.figure(figsize=(12, 4))
plt.plot(df["month"], znorm(df["instability"]),
         label="GDELT instability (z)", color="steelblue")
plt.plot(df["month"], znorm(df["psni_index"]),
         label="PSNI index (z)", color="coral", linestyle="--")
plt.axhline(0, color="black", linewidth=0.5)
plt.legend(); plt.title("GDELT vs PSNI instability measures")
plt.tight_layout(); plt.show()

## 3. Stormont suspension dummy (administrative record)
**Coding rule** (full justification in `docs/stormont_coding_note.md`):
a month is coded 1 if a functioning power-sharing Executive (First Minister
and deputy First Minister in office) was absent for the **majority of
calendar days** of that month. Caretaker ministers without FM/dFM do not
constitute a functioning Executive (no Executive Committee, no cross-cutting
or significant decisions). The Sept–Oct 2015 crisis is coded 0 (institutions
never formally collapsed). Sensitivity variants are estimated below.

In [ ]:
# Main specification
STORMONT_PERIODS = [
    ("2017-01", "2019-12"),   # McGuinness resignation (9 Jan 2017) -> NDNA (11 Jan 2020)
    ("2022-02", "2024-01"),   # Givan resignation (3 Feb 2022) -> restoration (3 Feb 2024)
]
# Sensitivity variants (boundary codings)
STORMONT_VARIANTS = {
    "inclusive_boundaries": [("2017-01", "2020-01"), ("2022-02", "2024-02")],
    "caretaker_as_functioning": [("2017-01", "2019-12"), ("2022-11", "2024-01")],
}

def stormont_dummy(data, periods):
    d = pd.Series(0, index=data.index)
    for s, e in periods:
        m = (data["month"] >= s) & (data["month"] <= pd.Period(e).to_timestamp("M"))
        d[m] = 1
    return d

df["stormont_suspended"] = stormont_dummy(df, STORMONT_PERIODS)
print("months suspended (main):", int(df["stormont_suspended"].sum()), "/", len(df))
for name, periods in STORMONT_VARIANTS.items():
    print(f"months suspended ({name}):",
          int(stormont_dummy(df, periods).sum()), "/", len(df))

## 4. Strict GKG filter (replicates the Events dual criterion)
Run once on BigQuery, save as `kg_theme_counts_northern_ireland_STRICT.csv`.

In [ ]:
KG_QUERY_STRICT = """
SELECT
  DATE_TRUNC(DATE(PARSE_TIMESTAMP('%Y%m%d%H%M%S',
             CAST(DATE AS STRING))), MONTH) AS month,
  theme,
  COUNT(*) AS mentions
FROM `gdelt-bq.gdeltv2.gkg`,
UNNEST(SPLIT(Themes, ';')) AS theme
WHERE EXISTS (
  SELECT 1
  FROM UNNEST(SPLIT(V2Locations, ';')) AS loc
  WHERE SPLIT(loc, '#')[SAFE_OFFSET(2)] = 'UK'
    AND SPLIT(loc, '#')[SAFE_OFFSET(1)] LIKE '%Northern Ireland%'
)
GROUP BY month, theme
ORDER BY month
"""
if RUN_BIGQUERY:
    output = DATA_DIR / "kg_theme_counts_northern_ireland_STRICT.csv"
    if output.exists() and not OVERWRITE_EXISTING:
        raise FileExistsError("Strict extract exists; set OVERWRITE_EXISTING to replace it.")
    if GCP_PROJECT_ID == "YOUR_GCP_PROJECT_ID":
        raise ValueError("Set GCP_PROJECT_ID before querying BigQuery.")
    if IN_COLAB:
        from google.colab import auth
        auth.authenticate_user()
    from google.cloud import bigquery
    client = bigquery.Client(project=GCP_PROJECT_ID)
    client.query(KG_QUERY_STRICT).to_dataframe().to_csv(output, index=False)
else:
    print("BigQuery disabled; using the local strict-filter CSV.")


In [ ]:
# Strict entropy vs original entropy
kg_strict = pd.read_csv(f"{BASE}/kg_theme_counts_northern_ireland_STRICT.csv",
                        parse_dates=["month"]).dropna(subset=["theme"])
nv_strict = (kg_strict.groupby("month")
             .apply(lambda x: entropy(x["mentions"], base=2))
             .rename("nv_strict").reset_index())
df = df.merge(nv_strict, on="month", how="left")

both = df[["nv", "nv_strict"]].dropna()
r_s, p_s = stats.pearsonr(both["nv"], both["nv_strict"])
print(f"corr(entropy loose, entropy strict): r={r_s:.3f} (p={p_s:.4g})")

plt.figure(figsize=(12, 4))
plt.plot(df["month"], znorm(df["nv"]), label="Entropy — original filter",
         color="steelblue")
plt.plot(df["month"], znorm(df["nv_strict"]), label="Entropy — strict filter",
         color="coral", linestyle="--")
plt.axhline(0, color="black", linewidth=0.5)
plt.legend(); plt.title("Loose vs strict GKG geographic filter")
plt.tight_layout(); plt.show()

## 5. Taxonomy drift: n_t trend test + Pielou evenness

In [ ]:
rich = (kg.groupby("month")
          .agg(n_themes=("theme", "nunique"),
               H=("mentions", lambda x: entropy(x, base=2)))
          .reset_index())
rich["pielou_J"] = rich["H"] / np.log2(rich["n_themes"])
df = df.merge(rich[["month", "n_themes", "pielou_J"]], on="month", how="left")

# sanity: recomputed H must equal stored nv
chk = df.merge(rich[["month", "H"]], on="month", how="left")
print("corr(nv, H recomputed):", round(chk["nv"].corr(chk["H"]), 6))

# trend test on n_t (OLS with HAC + Spearman)
t = np.arange(len(rich))
ols_n = sm.OLS(rich["n_themes"].values, sm.add_constant(t)).fit(
    cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS})
rho_n, p_rho_n = stats.spearmanr(t, rich["n_themes"].values)
print(f"n_t trend: beta={ols_n.params[1]:.3f} (p={ols_n.pvalues[1]:.4f}) | "
      f"Spearman rho={rho_n:.3f} (p={p_rho_n:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(rich["month"], rich["n_themes"], color="steelblue")
axes[0].set_title("Unique themes per month (n$_t$)")
axes[1].plot(df["month"], znorm(df["nv"]), label="Entropy (z)",
             color="steelblue")
axes[1].plot(df["month"], znorm(df["pielou_J"]), label="Pielou evenness (z)",
             color="coral", linestyle="--")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].legend(); axes[1].set_title("Entropy vs Pielou evenness")
plt.tight_layout(); plt.show()

## 6. Dynamic regressions M1–M5 (HAC errors)
- **M1** nv ~ nv₋₁ + PSNI₋₁ — media-independent predictor replaces GDELT
- **M2** nv ~ nv₋₁ + PSNI₋₁ + Stormont — adds institutional collapse
- **M3** nv ~ nv₋₁ + instability₋₁ + PSNI₋₁ — horse race
- **M4** Pielou ~ Pielou₋₁ + instability₋₁ — drift-corrected DV
- **M5** Pielou ~ Pielou₋₁ + PSNI₋₁ + Stormont — fully media-independent

In [ ]:
def dynreg(data, dv, predictors, label):
    d = data.copy()
    d[f"{dv}_lag1"] = d[dv].shift(1)
    for pr in predictors:
        if pr != "stormont_suspended":
            d[f"{pr}_lag1"] = d[pr].shift(1)
    cols = [f"{dv}_lag1"] + [
        pr if pr == "stormont_suspended" else f"{pr}_lag1"
        for pr in predictors]
    d = d.dropna(subset=[dv] + cols)
    res = sm.OLS(d[dv], sm.add_constant(d[cols].astype(float))).fit(
        cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS})
    out = pd.DataFrame({"coef": res.params, "se": res.bse,
                        "t": res.tvalues, "p": res.pvalues})
    out["model"], out["nobs"], out["adj_r2"] = label, int(res.nobs), res.rsquared_adj
    print(f"\n===== {label} (n={int(res.nobs)}, adj R2={res.rsquared_adj:.3f}) =====")
    print(out[["coef", "se", "p"]].round(4))
    return out

tables = pd.concat([
    dynreg(df, "nv", ["psni_index"], "M1_psni_only"),
    dynreg(df, "nv", ["psni_index", "stormont_suspended"], "M2_psni_stormont"),
    dynreg(df, "nv", ["instability", "psni_index"], "M3_horserace"),
    dynreg(df, "pielou_J", ["instability"], "M4_evenness_dv"),
    dynreg(df, "pielou_J", ["psni_index", "stormont_suspended"],
           "M5_evenness_psni"),
])
tables.to_csv(f"{RESULTS_DIR}/point2_dynamic_regressions.csv")

### 6b. Sensitivity: alternative Stormont boundary codings
Re-estimates M2 and M5 under both variants. Inspect the results before drawing conclusions about sensitivity.

In [ ]:
sens = []
for name, periods in STORMONT_VARIANTS.items():
    dv = df.copy()
    dv["stormont_suspended"] = stormont_dummy(dv, periods)
    sens.append(dynreg(dv, "nv", ["psni_index", "stormont_suspended"],
                       f"M2_{name}"))
    sens.append(dynreg(dv, "pielou_J", ["psni_index", "stormont_suspended"],
                       f"M5_{name}"))
pd.concat(sens).to_csv(f"{RESULTS_DIR}/point2_stormont_sensitivity.csv")

## 7. Local Projections with the PSNI shock (with Bonferroni and joint Wald tests)
The joint test also answers (multiple-testing correction).

In [ ]:
LAGS, HORIZONS = 3, 12
d7 = df[["month", "nv", "psni_index"]].dropna().reset_index(drop=True)
d7["shock"] = znorm(d7["psni_index"])
d7["y"] = znorm(d7["nv"])

rows = []
for h in range(HORIZONS + 1):
    tmp = pd.DataFrame({"yh": d7["y"].shift(-h),
                        "shock_l1": d7["shock"].shift(1)})
    for l in range(1, LAGS + 1):
        tmp[f"y_l{l}"] = d7["y"].shift(l)
        tmp[f"s_l{l}"] = d7["shock"].shift(l)
    tmp = tmp.dropna()
    X = tmp.drop(columns="yh").astype(float); X.insert(0, "const", 1.0)
    res = sm.OLS(tmp["yh"].astype(float), X).fit(
        cov_type="HAC", cov_kwds={"maxlags": max(h + 1, HAC_MAXLAGS)})
    b, se, pv = res.params["shock_l1"], res.bse["shock_l1"], res.pvalues["shock_l1"]
    rows.append({"h": h, "beta": b, "se": se, "p": pv,
                 "ci_lo": b - 1.96 * se, "ci_hi": b + 1.96 * se})
lp = pd.DataFrame(rows)
lp["p_bonferroni"] = (lp["p"] * len(lp)).clip(upper=1.0)
print(lp.round(4))
lp.to_csv(f"{RESULTS_DIR}/point2_lp_forward_psni.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(lp["h"], lp["beta"], "-o", color="steelblue")
plt.fill_between(lp["h"], lp["ci_lo"], lp["ci_hi"], alpha=0.2,
                 color="steelblue")
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Months after shock")
plt.ylabel("Response of NV (z) to 1-SD PSNI shock")
plt.title("Local Projections — media-independent instability shock")
plt.tight_layout(); plt.show()

# joint Wald test: shock lags 1..6 jointly zero (distributed-lag regression)
tmpJ = pd.DataFrame({"y": d7["y"]})
shock_cols = []
for l in range(1, 7):
    c = f"shock_l{l}"; tmpJ[c] = d7["shock"].shift(l); shock_cols.append(c)
for l in range(1, LAGS + 1):
    tmpJ[f"y_l{l}"] = d7["y"].shift(l)
tmpJ = tmpJ.dropna()
XJ = tmpJ.drop(columns="y").astype(float); XJ.insert(0, "const", 1.0)
resJ = sm.OLS(tmpJ["y"].astype(float), XJ).fit(
    cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS})
fJ = resJ.f_test(" = 0, ".join(shock_cols) + " = 0")
print(f"Joint Wald (shock lags 1-6): F={float(fJ.fvalue):.2f} "
      f"p={float(fJ.pvalue):.4f} (n={int(resJ.nobs)})")

## 8. Summary of results
- Theme richness `n_t` declines significantly over the sample (trend p < 0.001, Spearman rho = -0.63): taxonomy drift biases entropy downward, and Pielou evenness leaves the pattern of results unchanged.
- Strict versus baseline Knowledge Graph filter: the two fragmentation series correlate at r = 0.998; the filtering asymmetry does not drive the findings.
- corr(GDELT instability, PSNI index) = 0.12: recorded security incidents capture a dimension largely orthogonal to institutional and constitutional instability.
- M1 to M5: neither the PSNI index nor the suspension indicator significantly predicts subsequent fragmentation; the negative suspension coefficient is consistent with agenda consolidation.
- These results are reported in Section 4.7 and Table 4 of the paper.
